In [1]:
import pyspark
from pyspark.sql.functions import monotonically_increasing_id
from pyspark.sql import SparkSession
import os

In [2]:
spark = (SparkSession.builder
         .appName("MySpark")
         .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
         .master("local[*]")
         .getOrCreate())

In [3]:
pg_user = os.getenv("POSTGRES_USER", "postgres")
pg_password = os.getenv("POSTGRES_PASSWORD")
pg_host = os.getenv("POSTGRES_HOST", "postgres")
pg_port = os.getenv("POSTGRES_PORT", 5432)
pg_db = "postgres"

jdbc_url = f"jdbc:postgresql://{pg_host}:{pg_port}/{pg_db}"

base_reader = (spark.read
               .format("jdbc")
               .option("url", jdbc_url)
               .option("driver", "org.postgresql.Driver")
               .option("user", pg_user)
               .option("password", pg_password))


In [4]:
mock_data_db = base_reader.option("dbtable", "public.mock_data").load()

In [60]:
def write_to_table_pgsql(df, table):
    (df.write
       .format("jdbc")
       .option("url", jdbc_url)
       .option("driver", "org.postgresql.Driver")
       .option("user", pg_user)
       .option("password", pg_password)
       .option("dbtable", table)
       .mode("append")
       .save())

In [77]:
dim_customers = mock_data_db[['customer_first_name', 'customer_last_name', 'customer_age',
    'customer_email', 'customer_country', 'customer_postal_code',
    'customer_pet_type', 'customer_pet_name', 'customer_pet_breed', 'pet_category']].withColumn('id', monotonically_increasing_id() + 1)

dim_products = mock_data_db[['product_name', 'product_category', 'product_price', 'product_quantity',
    'product_weight', 'product_color', 'product_size', 'product_brand',
    'product_material', 'product_description', 'product_rating',
    'product_reviews', 'product_release_date', 'product_expiry_date']].withColumn('id', monotonically_increasing_id() + 1)

dim_sellers = mock_data_db[['seller_first_name', 'seller_last_name', 'seller_email',
    'seller_country', 'seller_postal_code']].withColumn('id', monotonically_increasing_id() + 1)

dim_stores = mock_data_db[['store_name', 'store_location', 'store_city', 'store_state',
    'store_country', 'store_phone', 'store_email']].withColumn('id', monotonically_increasing_id() + 1)

dim_suppliers = mock_data_db[['supplier_name', 'supplier_contact', 'supplier_email', 'supplier_phone',
    'supplier_address', 'supplier_city', 'supplier_country']].withColumn('id', monotonically_increasing_id() + 1)

fact_sales = mock_data_db[['sale_date', 'sale_customer_id', 'sale_seller_id', 'sale_product_id',
    'sale_quantity', 'sale_total_price']].withColumn('id', monotonically_increasing_id() + 1)

In [7]:
dim_countries = dim_customers.select(['customer_country'])
dim_countries = dim_countries.union(dim_sellers.select(['seller_country']))
dim_countries = dim_countries.union(dim_suppliers.select(['supplier_country']))
dim_countries = dim_countries.union(dim_stores.select(['store_country']))

In [8]:
dim_countries = dim_countries.withColumnRenamed('customer_country', 'name').distinct().withColumn('id', monotonically_increasing_id() + 1)

In [63]:
write_to_table_pgsql(dim_countries, 'dim_countries')

In [10]:
dim_countries.show(5)

+-------------------+---+
|               name| id|
+-------------------+---+
|               Chad|  1|
|             Russia|  2|
|           Paraguay|  3|
|              Yemen|  4|
|U.S. Virgin Islands|  5|
+-------------------+---+
only showing top 5 rows



In [48]:
dim_suppliers = dim_suppliers.join(dim_countries.withColumnRenamed('id', 'country_id'), dim_countries['name'] == dim_suppliers['supplier_country']).drop('name', 'supplier_country').orderBy('id')

dim_customers = dim_customers.join(dim_countries.withColumnRenamed('id', 'country_id'), dim_countries['name'] == dim_customers['customer_country']).drop('name', 'customer_country').orderBy('id')

dim_stores = dim_stores.join(dim_countries.withColumnRenamed('id', 'country_id'), dim_countries['name'] == dim_stores['store_country']).drop('name', 'store_country').orderBy('id')

dim_sellers = dim_sellers.join(dim_countries.withColumnRenamed('id', 'country_id'), dim_countries['name'] == dim_sellers['seller_country']).drop('name', 'seller_country').orderBy('id')

In [37]:
dim_pet_categories = dim_customers[['pet_category']].distinct().withColumn('id', monotonically_increasing_id() + 1)

In [64]:
write_to_table_pgsql(dim_pet_categories.withColumnRenamed('pet_category', 'category'), 'dim_pet_categories')
dim_pet_categories.show()

+------------+---+
|pet_category| id|
+------------+---+
|    Reptiles|  1|
|        Fish|  2|
|        Dogs|  3|
|       Birds|  4|
|        Cats|  5|
+------------+---+



In [49]:
dim_customers = dim_customers.join(dim_pet_categories.withColumnRenamed('id', 'pet_category_id'), dim_pet_categories['pet_category'] == dim_customers['pet_category']).drop('pet_category')

dim_customers.show(5)

+-------------------+------------------+------------+--------------------+--------------------+-----------------+-----------------+------------------+----+----------+---------------+
|customer_first_name|customer_last_name|customer_age|      customer_email|customer_postal_code|customer_pet_type|customer_pet_name|customer_pet_breed|  id|country_id|pet_category_id|
+-------------------+------------------+------------+--------------------+--------------------+-----------------+-----------------+------------------+----+----------+---------------+
|              Lanae|            Seefus|          69|dnorebi@parallels...|                NULL|              dog|            Storm|Labrador Retriever|3443|         1|              1|
|             Giorgi|        Glendining|          84|kmarushak24@bing.com|                NULL|             bird|             Bard|Labrador Retriever|8097|         1|              1|
|             Emmott|          Capstack|          43|slytlle20@census.gov|           

In [79]:
sellers_with_store_emails = dim_sellers.join(mock_data_db.select('store_email', 'seller_email'), 'seller_email')

dim_sellers = sellers_with_store_emails.join(dim_stores.select('id', 'store_email').withColumnRenamed('id', 'seller_store_id'), 'store_email').drop('store_email')

In [51]:
products_with_email = dim_products.join(mock_data_db.select('product_brand', 'product_name', 'product_quantity', 'supplier_email'),
                  ['product_brand', 'product_name', 'product_quantity'])

dim_products = products_with_email.join(dim_suppliers.select('supplier_email', 'id').withColumnRenamed('id', 'product_supplier_id'),
                         products_with_email['supplier_email'] == dim_suppliers['supplier_email']).drop('supplier_email')

In [52]:
dim_products.show()

+-------------+------------+----------------+----------------+-------------+--------------+-------------+------------+----------------+--------------------+--------------+---------------+--------------------+-------------------+----+-------------------+
|product_brand|product_name|product_quantity|product_category|product_price|product_weight|product_color|product_size|product_material| product_description|product_rating|product_reviews|product_release_date|product_expiry_date|  id|product_supplier_id|
+-------------+------------+----------------+----------------+-------------+--------------+-------------+------------+----------------+--------------------+--------------+---------------+--------------------+-------------------+----+-------------------+
|     Realcube|   Bird Cage|              31|            Cage|        67.69|          45.3|       Fuscia|       Large|           Brass|Praesent id massa...|           2.6|            892|          10/20/2022|         10/29/2027|5630|     

In [66]:
dim_products.select('id').distinct().count()

10000

In [90]:
write_to_table_pgsql(dim_customers, 'dim_customers')

write_to_table_pgsql(dim_suppliers, 'dim_suppliers')

write_to_table_pgsql(dim_products, 'dim_products')

write_to_table_pgsql(dim_stores, 'dim_stores')

write_to_table_pgsql(dim_sellers, 'dim_sellers')

write_to_table_pgsql(fact_sales, 'fact_sales')